Analiza i prognozowanie zapotrzebowania na moc w Krajowym Systemie Elektroenergetycznym (KSE) - na przykładzie Niemiec

**Autor:** Marek Burkot

In [1]:
# ==========================================
# 1. LIBRARIES IMPORT
# ==========================================
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger("chronos").setLevel(logging.ERROR)

import pandas as pd
import numpy as np
import holidays
import torch
import plotly.graph_objects as go
import time
from IPython.display import display

# Sklearn - classic ML and metrics
from sklearn.model_selection import train_test_split, RandomizedSearchCV, TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Gradient boosting algorithms
import xgboost as xgb
import lightgbm as lgb

# Keras - Deep Learning (LSTM / GRU)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

# Fundamental model (Zero-Shot)
from chronos import ChronosPipeline

## 1. Wczytywanie i czyszczenie danych (Data Cleaning)
Wstępna obróbka danych z ENTSO-E oraz danych pogodowych dla Niemiec.

In [2]:
# ==========================================
# 2. DATA CLEANING
# ==========================================

print("Loading and joining data")
df_power = pd.read_parquet('data/time_series_60min_singleindex.parquet')
df_weather = pd.read_parquet('data/weather_data.parquet')

df_power['utc_timestamp'] = pd.to_datetime(df_power['utc_timestamp'])
df_weather['utc_timestamp'] = pd.to_datetime(df_weather['utc_timestamp'])
df_power.set_index('utc_timestamp', inplace=True)
df_weather.set_index('utc_timestamp', inplace=True)

power_cols = [col for col in df_power.columns if col.startswith('DE_')]
weather_cols = [col for col in df_weather.columns if col.startswith('DE_')]
df_de_power = df_power[power_cols].copy()
df_de_weather = df_weather[weather_cols].copy()

df = df_de_power.join(df_de_weather, how='inner')
df = df[df.index >= '2015-01-01']

drop_prefixes_regions = ["DE_50hertz", "DE_amprion", "DE_tennet", "DE_transnetbw", "DE_LU"]
columns_to_drop_reg = [col for col in df.columns if any(col.startswith(prefix) for prefix in drop_prefixes_regions)]
df.drop(columns_to_drop_reg, axis=1, inplace=True)

drop_keywords = ['capacity', 'profile', 'offshore', 'onshore']
columns_to_drop_key = [col for col in df.columns if any(keyword in col for keyword in drop_keywords)]
df.drop(columns_to_drop_key, axis=1, inplace=True)

rename_dict = {
    'DE_load_actual_entsoe_transparency': 'Load_Target',
    'DE_load_forecast_entsoe_transparency': 'Load_Forecast',
    'DE_solar_generation_actual': 'Solar_Gen',
    'DE_wind_generation_actual': 'Wind_Gen',
    'DE_temperature': 'Temperature',
    'DE_radiation_direct_horizontal': 'Radiation_Direct',
    'DE_radiation_diffuse_horizontal': 'Radiation_Diffuse'
}
df.rename(columns=rename_dict, inplace=True)

df = df.interpolate(method='time')
df = df.bfill()

print(f"Data ready. Shape: {df.shape}")

Loading and joining data
Data ready. Shape: (43824, 7)


## 2. Inżynieria Cech (Feature Engineering)
Ekstrakcja zmiennych kalendarzowych, flagowanie dni ustawowo wolnych od pracy oraz tworzenie opóźnień (Lags).

In [3]:
# ==========================================
# 3. FEATURE ENGINEERING & TRAIN/TEST SPLIT
# ==========================================
def create_features(data):
    df_feat = data.copy()

    # Basic calendar attributes
    df_feat['Hour'] = df_feat.index.hour
    df_feat['DayOfWeek'] = df_feat.index.dayofweek
    df_feat['Month'] = df_feat.index.month
    df_feat['IsWeekend'] = df_feat['DayOfWeek'].isin([5, 6]).astype(int)

    # Holidays for Germany
    years = df_feat.index.year.unique().tolist()
    de_holidays = holidays.DE(years=years)
    df_feat['IsHoliday'] = df_feat.index.normalize().isin(de_holidays).astype(int)

    # Identify the two most common bridge day patterns:
    # A) Friday after a Thursday holiday (e.g., Corpus Christi, Ascension)
    is_friday = df_feat['DayOfWeek'] == 4
    was_thurs_holiday = (df_feat.index.normalize() - pd.Timedelta(days=1)).isin(de_holidays)

    # B) Monday before a Tuesday holiday
    is_monday = df_feat['DayOfWeek'] == 0
    will_tues_holiday = (df_feat.index.normalize() + pd.Timedelta(days=1)).isin(de_holidays)

    # Combine into a single flag
    df_feat['IsBridgeDay'] = (is_friday & was_thurs_holiday) | (is_monday & will_tues_holiday)
    df_feat['IsBridgeDay'] = df_feat['IsBridgeDay'].astype(int)

    # 2. Holiday Proximity (DaysUntilHoliday) - Countdown to the next day off
    # Calculates how many days are left until the next holiday (0 = today is a holiday).
    holiday_dates = pd.to_datetime(list(de_holidays.keys())).values.astype('datetime64[D]')
    unique_dates = df_feat.index.normalize().unique().values.astype('datetime64[D]')

    until_dict = {} 
    for d in unique_dates:
        future_holidays = holiday_dates[holiday_dates >= d]
        if len(future_holidays) > 0:
            until_dict[d] = int((np.min(future_holidays) - d) / np.timedelta64(1, 'D'))
        else:
            until_dict[d] = 999

    df_feat['DaysUntilHoliday'] = df_feat.index.normalize().values.astype('datetime64[D]')
    df_feat['DaysUntilHoliday'] = df_feat['DaysUntilHoliday'].map(until_dict)

    # Lagged Variables
    df_feat['LoadLag24h'] = df_feat['Load_Target'].shift(24)
    df_feat['LoadLag168h'] = df_feat['Load_Forecast'].shift(168)

    return df_feat

df = create_features(df)
df.dropna(inplace=True) # Remove empty rows resulting from the 168h shift

X = df.drop(columns=['Load_Target', 'Load_Forecast'])
y = df['Load_Target']
entsoe_forecast = df['Load_Forecast']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
_, entsoe_test = train_test_split(entsoe_forecast, test_size=0.2, shuffle=False)

print("Feature Engineering 2.0 successfully applied!")
print(f"Added new features: IsBridgeDay, DaysUntilHoliday")
print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

results = []

Feature Engineering 2.0 successfully applied!
Added new features: IsBridgeDay, DaysUntilHoliday
Training set: (34924, 14)
Testing set: (8732, 14)


## 3. Klasyczne modele Uczenia Maszynowego i Ensemble Learning
Analiza porównawcza modeli liniowych, drzew decyzyjnych oraz algorytmów z rodziny Gradient Boostingu.

In [4]:
# ==========================================
# 4. CLASSIC ML TRAINING
# ==========================================

models = {
    "1. Linear Regression": LinearRegression(),
    "2. Decision Tree": DecisionTreeRegressor(max_depth=20, random_state=42),
    "3. Random Forest (Bagging)": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "4. LightGBM (Boosting)": lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1),
    "5. XGBoost (Boosting)": xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

for name, model in models.items():
    print(f"Training: {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100

    results.append({"Model": name, "MAE [MW]": round(mae, 2), "RMSE [MW]": round(rmse, 2), "MAPE [%]": round(mape, 2)})

mae_entsoe = mean_absolute_error(y_test, entsoe_test)
rmse_entsoe = np.sqrt(mean_squared_error(y_test, entsoe_test))
mape_entsoe = mean_absolute_percentage_error(y_test, entsoe_test) * 100
results.append({"Model": "0. ENTSO-E (Benchmark)", "MAE [MW]": round(mae_entsoe, 2), "RMSE [MW]": round(rmse_entsoe, 2), "MAPE [%]": round(mape_entsoe, 2)})

print("\nEnd of classic ML training.")

Training: 1. Linear Regression...
Training: 2. Decision Tree...
Training: 3. Random Forest (Bagging)...
Training: 4. LightGBM (Boosting)...
Training: 5. XGBoost (Boosting)...

End of classic ML training.


## 4. Modele Głębokiego Uczenia (Deep Learning)
Eksperymenty z architekturami rekurencyjnymi (RNN) - zastosowanie sieci GRU oraz LSTM z optymalizacją hiperparametrów.

In [5]:
# ==========================================
# 5. DEEP LEARNING (LSTM & GRU)
# ==========================================

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

TIME_STEPS = 24

def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

X_train_3d, y_train_3d = create_sequences(X_train_scaled, y_train_scaled, TIME_STEPS)
X_test_3d, y_test_3d = create_sequences(X_test_scaled, y_test_scaled, TIME_STEPS)
y_test_actual = y_test.iloc[TIME_STEPS:].values

print("LSTM Training...")
model_lstm = Sequential([
    Input(shape=(X_train_3d.shape[1], X_train_3d.shape[2])),
    LSTM(128, return_sequences=True),
    BatchNormalization(),
    Dropout(0.2),
    LSTM(64),
    Dense(1)
])
model_lstm.compile(optimizer=Adam(0.001), loss='mse')
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

model_lstm.fit(X_train_3d, y_train_3d, epochs=30, batch_size=32, validation_split=0.1, callbacks=[early_stop], verbose=0)
y_pred_lstm = scaler_y.inverse_transform(model_lstm.predict(X_test_3d, verbose=0)).flatten()

print("GRU Training...")
model_gru = Sequential([
    Input(shape=(X_train_3d.shape[1], X_train_3d.shape[2])),
    GRU(128, return_sequences=True),
    BatchNormalization(),
    Dropout(0.2),
    GRU(64),
    Dense(1)
])
model_gru.compile(optimizer=Adam(0.001), loss='mse')

model_gru.fit(X_train_3d, y_train_3d, epochs=30, batch_size=32, validation_split=0.1, callbacks=[early_stop], verbose=0)
y_pred_gru = scaler_y.inverse_transform(model_gru.predict(X_test_3d, verbose=0)).flatten()

for name, y_pred_dl in [("6. LSTM", y_pred_lstm), ("7. GRU", y_pred_gru)]:
    mae = mean_absolute_error(y_test_actual, y_pred_dl)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_dl))
    mape = mean_absolute_percentage_error(y_test_actual, y_pred_dl) * 100
    results.append({"Model": name, "MAE [MW]": round(mae, 2), "RMSE [MW]": round(rmse, 2), "MAPE [%]": round(mape, 2)})

print("End of RNN models training")

LSTM Training...
GRU Training...
End of RNN models training


## 5. Eksperyment Zero-Shot (Foundation Models)
Testowanie potężnego pretrenowanego modelu dla szeregów czasowych (Amazon Chronos-T5) pracującego bez wiedzy domenowej (Features).

In [6]:
# ==========================================
# 6. AMAZON CHRONOS (ZERO-SHOT AI)
# ==========================================
print("Initialization of Amazon Chronos-T5")

context_length = 512
horizon_length = 24

context_data = y_test.iloc[-context_length - horizon_length : -horizon_length].values
actual_24h = y_test.iloc[-horizon_length:].values
context_tensor = torch.tensor(context_data, dtype=torch.float32)

pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-base",
    device_map="cpu",
    dtype=torch.float32,
)

print("Predict generation Zero-Shot...")
forecast = pipeline.predict(context_tensor, prediction_length=horizon_length)
y_pred_chronos = np.quantile(forecast[0].numpy(), 0.5, axis=0)

mae_chronos = mean_absolute_error(actual_24h, y_pred_chronos)
rmse_chronos = np.sqrt(mean_squared_error(actual_24h, y_pred_chronos))
mape_chronos = mean_absolute_percentage_error(actual_24h, y_pred_chronos) * 100

results.append({
    "Model": "8. Amazon Chronos-T5 (Zero-Shot)",
    "MAE [MW]": round(mae_chronos, 2),
    "RMSE [MW]": round(rmse_chronos, 2),
    "MAPE [%]": round(mape_chronos, 2)
})
print("End of Amazon Chronos-T5 training.")

Initialization of Amazon Chronos-T5
Predict generation Zero-Shot...
End of Amazon Chronos-T5 training.


## 6. Ostateczne zestawienie wyników badawczych

In [7]:
# ==========================================
# 7. SCORES
# ==========================================
df_results = pd.DataFrame(results).sort_values(by="MAPE [%]").reset_index(drop=True)
display(df_results)

,Model,MAE [MW],RMSE [MW],MAPE [%]
0,3. Random Forest (Bagging),1161.13,1728.92,2.15
1,5. XGBoost (Boosting),1190.39,1735.84,2.20
2,4. LightGBM (Boosting),1201.94,1765.08,2.22
3,6. LSTM,1210.64,1982.80,2.26
4,2. Decision Tree,1546.38,2353.53,2.85
5,0. ENTSO-E (Benchmark),1989.40,2488.29,3.52
6,7. GRU,2678.11,3616.15,4.97
7,1. Linear Regression,2781.51,4747.49,5.22
8,8. Amazon Chronos-T5 (Zero-Shot),3864.75,4381.53,8.08


## 7. Analiza analityczna i wizualna anomalii rynkowych
Zjawisko wpływu kalendarza (Feature Engineering) vs. zdolności predykcyjne AI.

In [8]:
# ==========================================
# 8. PLOTLY VISUALIZATION
# ==========================================
y_pred_lr = models["1. Linear Regression"].predict(X_test)
y_pred_rf = models["3. Random Forest (Bagging)"].predict(X_test)

series_actual = y_test
series_entsoe = entsoe_test
series_lr = pd.Series(y_pred_lr, index=y_test.index)
series_rf = pd.Series(y_pred_rf, index=y_test.index)
series_lstm = pd.Series(y_pred_lstm, index=y_test.index[TIME_STEPS:])

def plot_period(start_date, end_date, title):
    mask_full = (series_actual.index >= start_date) & (series_actual.index <= end_date)
    mask_lstm = (series_lstm.index >= start_date) & (series_lstm.index <= end_date)

    context_data_chronos = series_actual[series_actual.index < start_date].tail(512).values
    horizon_length_chronos = int(mask_full.sum())

    if len(context_data_chronos) > 0:
        context_tensor = torch.tensor(context_data_chronos, dtype=torch.float32)
        forecast = pipeline.predict(context_tensor, prediction_length=horizon_length_chronos)
        y_pred_chronos_window = np.quantile(forecast[0].numpy(), 0.5, axis=0)
        series_chronos = pd.Series(y_pred_chronos_window, index=series_actual[mask_full].index)

    fig = go.Figure()

    # 1. Reality (Black, thick baseline)
    fig.add_trace(go.Scatter(x=series_actual[mask_full].index, y=series_actual[mask_full],
                             mode='lines', name='Rzeczywiste obciążenie',
                             line=dict(color='black', width=3.5)))

    # 2. ENTSO-E Benchmark (Official Operator Forecast - Blue, dashed)
    fig.add_trace(go.Scatter(x=series_entsoe[mask_full].index, y=series_entsoe[mask_full],
                             mode='lines', name='ENTSO-E (Operator)',
                             line=dict(color='#119DFF', width=2, dash='dash')))

    # 3. Linear Regression (Orange)
    fig.add_trace(go.Scatter(x=series_lr[mask_full].index, y=series_lr[mask_full],
                             mode='lines', name='Regresja Liniowa',
                             line=dict(color='#FFA15A', width=2)))
    # 4. Random Forest (Emerald Green)
    fig.add_trace(go.Scatter(x=series_rf[mask_full].index, y=series_rf[mask_full],
                             mode='lines', name='Random Forest',
                             line=dict(color='#00CC96', width=2)))

    # 5. LSTM (Deep Red)
    fig.add_trace(go.Scatter(x=series_lstm[mask_lstm].index, y=series_lstm[mask_lstm],
                             mode='lines', name='LSTM (Deep Learning)',
                             line=dict(color='#EF553B', width=2)))

    # 6. Amazon Chronos (Purple, dashdot)
    if len(context_data_chronos) > 0:
        fig.add_trace(go.Scatter(x=series_chronos.index, y=series_chronos,
                                 mode='lines', name='Amazon Chronos',
                                 line=dict(color='#AB63FA', width=2.5, dash='dashdot')))

    fig.update_layout(
        title=f'<b>{title}</b>',
        xaxis_title='Date',
        yaxis_title='Power Demand [MW]',
        hovermode='x unified',
        template='plotly_white',
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
            font=dict(size=12), bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=1
        ),
        margin=dict(l=40, r=40, t=80, b=40)
    )
    fig.show()

plot_period('2019-12-23', '2019-12-27', 'Scenariusz A: Załamanie popytu (Boże Narodzenie)')
plot_period('2019-05-29', '2019-06-02', 'Scenariusz B: Dzień pomostowy (Majówka)')
plot_period('2019-07-18', '2019-07-22', 'Scenariusz C: Standardowy dzień (Lipiec)')

We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


In [9]:
# ==========================================
# 9. HYPERPARAMETERS TUNING
# ==========================================
tuned_results = []

# Creating a chronological split for time series (prevents data leakage!)
tscv = TimeSeriesSplit(n_splits=3)

# ---------------------------------------------------------
# 9.1 XGBoost
# ---------------------------------------------------------
print("\n--- Tuning XGBoost ---")
xgb_base = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

xgb_param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 6, 8],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_grid,
    n_iter=10,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
xgb_random_search.fit(X_train, y_train)
xgb_tune_time = time.time() - start_time

print(f"Best parameters: {xgb_random_search.best_params_}")

best_xgb = xgb_random_search.best_estimator_
y_pred_xgb_tuned = best_xgb.predict(X_test)

tuned_results.append({
    "Model": "XGBoost (Tuned)",
    "MAE [MW]": round(mean_absolute_error(y_test, y_pred_xgb_tuned), 2),
    "RMSE [MW]": round(np.sqrt(mean_squared_error(y_test, y_pred_xgb_tuned)), 2),
    "MAPE [%]": round(mean_absolute_percentage_error(y_test, y_pred_xgb_tuned) * 100, 2),
    "Tuning Time [s]": round(xgb_tune_time, 2)
})

# ---------------------------------------------------------
# 9.2 LightGBM
# ---------------------------------------------------------
print("\n--- Tuning LightGBM ---")
lgb_base = lgb.LGBMRegressor(random_state=42, verbose=-1)

lgb_param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63, 127],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

lgb_random_search = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=lgb_param_grid,
    n_iter=10,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
lgb_random_search.fit(X_train, y_train)
lgb_tune_time = time.time() - start_time
print(f"Best parameters: {lgb_random_search.best_params_}")

best_lgb = lgb_random_search.best_estimator_
y_pred_lgb_tuned = best_lgb.predict(X_test)

tuned_results.append({
    "Model": "LightGBM (Tuned)",
    "MAE [MW]": round(mean_absolute_error(y_test, y_pred_lgb_tuned), 2),
    "RMSE [MW]": round(np.sqrt(mean_squared_error(y_test, y_pred_lgb_tuned)), 2),
    "MAPE [%]": round(mean_absolute_percentage_error(y_test, y_pred_lgb_tuned) * 100, 2),
    "Tuning Time [s]": round(lgb_tune_time, 2)
})

# ---------------------------------------------------------
# 9.3 Random Forest
# ---------------------------------------------------------
print("\n--- Tuning Random Forest ---")
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_grid,
    n_iter=10,
    scoring='neg_root_mean_squared_error',
    cv=tscv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
rf_random_search.fit(X_train, y_train)
rf_tune_time = time.time() - start_time

print(f"Best parameters: {rf_random_search.best_params_}")

best_rf = rf_random_search.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test)

tuned_results.append({
    "Model": "Random Forest (Tuned)",
    "MAE [MW]": round(mean_absolute_error(y_test, y_pred_rf_tuned), 2),
    "RMSE [MW]": round(np.sqrt(mean_squared_error(y_test, y_pred_rf_tuned)), 2),
    "MAPE [%]": round(mean_absolute_percentage_error(y_test, y_pred_rf_tuned) * 100, 2),
    "Tuning Time [s]": round(rf_tune_time, 2)
})


--- Tuning XGBoost ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters: {'subsample': 1.0, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

--- Tuning LightGBM ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters: {'subsample': 1.0, 'num_leaves': 63, 'n_estimators': 300, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

--- Tuning Random Forest ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters: {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 30}


In [10]:
print("\n================ RESULTS AFTER OPTIMIZATION ================")
df_tuned = pd.DataFrame(tuned_results)
display(df_tuned)


================ RESULTS AFTER OPTIMIZATION ================


,Model,MAE [MW],RMSE [MW],MAPE [%],Tuning Time [s]
0,XGBoost (Tuned),1056.45,1606.82,1.96,13.01
1,LightGBM (Tuned),1049.01,1586.44,1.95,12.81
2,Random Forest (Tuned),1158.24,1724.26,2.14,29.59


In [11]:
# ==========================================
# 10. FINAl VISUALIZATION FOR TUNED MODELS
# ==========================================

series_actual = y_test
series_entsoe = entsoe_test
series_xgb_t = pd.Series(y_pred_xgb_tuned, index=y_test.index)
series_lgb_t = pd.Series(y_pred_lgb_tuned, index=y_test.index)
series_rf_t = pd.Series(y_pred_rf_tuned, index=y_test.index)

def plot_period(start_date, end_date, title):
    mask = (series_actual.index >= start_date) & (series_actual.index <= end_date)
    fig = go.Figure()

    # 1. Reality (Black, thick baseline)
    fig.add_trace(go.Scatter(x=series_actual[mask].index, y=series_actual[mask],
                             mode='lines', name='Actual Load',
                             line=dict(color='black', width=3.5)))

    # 2. ENTSO-E Benchmark (Official operator forecast - Blue, dashed)
    fig.add_trace(go.Scatter(x=series_entsoe[mask].index, y=series_entsoe[mask],
                             mode='lines', name='ENTSO-E (Operator)',
                             line=dict(color='#119DFF', width=2, dash='dash')))

    # 3. XGBoost Tuned (Vivid Red)
    fig.add_trace(go.Scatter(x=series_xgb_t[mask].index, y=series_xgb_t[mask],
                             mode='lines', name='XGBoost (Tuned)',
                             line=dict(color='#EF553B', width=2)))

    # 4. LightGBM Tuned (Purple)
    fig.add_trace(go.Scatter(x=series_lgb_t[mask].index, y=series_lgb_t[mask],
                             mode='lines', name='LightGBM (Tuned)',
                             line=dict(color='#AB63FA', width=2)))

    # 5. Random Forest Tuned (Emerald Green)
    fig.add_trace(go.Scatter(x=series_rf_t[mask].index, y=series_rf_t[mask],
                             mode='lines', name='Random Forest (Tuned)',
                             line=dict(color='#00CC96', width=2)))

    fig.update_layout(
        title=f'<b>{title}</b>',
        xaxis_title='Date',
        yaxis_title='Power Demand [MW]',
        hovermode='x unified',
        template='plotly_white',
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1,
            font=dict(size=12), bgcolor="rgba(255,255,255,0.9)", bordercolor="black", borderwidth=1
        ),
        margin=dict(l=40, r=40, t=80, b=40)
    )
    fig.show()


plot_period('2019-12-24', '2019-12-28', 'Scenariusz A: załamanie popytu (Boże Narodzenie)')
plot_period('2019-05-29', '2019-06-02', 'Scenariusz B: dzień pomostowy (Majówka)')
plot_period('2019-07-18', '2019-07-21', 'Scenariusz C: standardowy dzień (Lipiec)')

In [12]:
# ==========================================
# 11. VISUALIZATIONS FOR .DOCX
# ==========================================

y_pred_xgb = best_xgb.predict(X_test)
y_pred_lgb = best_lgb.predict(X_test)

series_actual = y_test
series_entsoe = entsoe_test
series_xgb_tuned = pd.Series(y_pred_xgb, index=y_test.index)
series_lgb_tuned = pd.Series(y_pred_lgb, index=y_test.index)

def plot_period_optimized(start_date, end_date, title):
    mask_full = (series_actual.index >= start_date) & (series_actual.index <= end_date)

    fig = go.Figure()

    # 1. Reality (Black, thick baseline)
    fig.add_trace(go.Scatter(x=series_actual[mask_full].index, y=series_actual[mask_full],
                             mode='lines', name='Rzeczywiste zapotrzebowanie',
                             line=dict(color='black', width=3.5)))

    # 2. ENTSO-E Benchmark (Official operator forecast - Blue, dashed)
    fig.add_trace(go.Scatter(x=series_entsoe[mask_full].index, y=series_entsoe[mask_full],
                             mode='lines', name='Prognoza operatora',
                             line=dict(color='#119DFF', width=2.5, dash='dash')))

    # 3. XGBoost (Vivid Red)
    fig.add_trace(go.Scatter(x=series_xgb_tuned[mask_full].index, y=series_xgb_tuned[mask_full],
                             mode='lines', name='XGBoost',
                             line=dict(color='#EF553B', width=2.5)))
                             
    # 4. LightGBM (Purple)
    fig.add_trace(go.Scatter(x=series_lgb_tuned[mask_full].index, y=series_lgb_tuned[mask_full],
                             mode='lines', name='LightGBM',
                             line=dict(color='#AB63FA', width=2.5)))

    fig.update_layout(
        title=dict(text=f'<b>{title}</b>', font=dict(size=18)),
        xaxis_title=dict(text='Data', font=dict(size=14)),
        yaxis_title=dict(text='Zapotrzebowanie na moc [MW]', font=dict(size=14)),
        font=dict(size=12),
        width=850,
        height=580,
        hovermode='x unified',
        template='plotly_white',
        legend=dict(
            orientation="h", 
            yanchor="top", y=-0.25,
            xanchor="center", x=0.5,
            font=dict(size=12), 
            bgcolor="rgba(255,255,255,0.9)", 
            bordercolor="black", borderwidth=1
        ),
        margin=dict(l=60, r=40, t=60, b=120)
    )

    fig.update_xaxes(
        tickformat="%d%b %H:00"
    )
    
    fig.show()

plot_period_optimized('2019-12-24', '2019-12-28', 'Scenariusz A: Załamanie popytu (Boże Narodzenie)')

In [13]:
y_pred_xgb = best_xgb.predict(X_test)
y_pred_lgb = best_lgb.predict(X_test)
y_pred_rf = best_rf.predict(X_test)

series_actual = y_test
series_entsoe = entsoe_test
series_xgb_tuned = pd.Series(y_pred_xgb, index=y_test.index)
series_lgb_tuned = pd.Series(y_pred_lgb, index=y_test.index)
series_rf_tuned = pd.Series(y_pred_rf, index=y_test.index)

def plot_period_residuals(start_date, end_date, title):
    mask_full = (series_actual.index >= start_date) & (series_actual.index <= end_date)

    dev_actual = series_actual[mask_full] - series_actual[mask_full] 
    dev_entsoe = series_entsoe[mask_full] - series_actual[mask_full]
    dev_xgb = series_xgb_tuned[mask_full] - series_actual[mask_full]
    dev_lgb = series_lgb_tuned[mask_full] - series_actual[mask_full]

    fig = go.Figure()

    # 1. Reality (Black, thick baseline) - as 0
    fig.add_trace(go.Scatter(x=dev_actual.index, y=dev_actual,
                             mode='lines', name='Rzeczywiste zapotrzebowanie',
                             line=dict(color='black', width=4)))

    # 2. ENTSO-E Benchmark (Official operator forecast - Blue, dashed)
    fig.add_trace(go.Scatter(x=dev_entsoe.index, y=dev_entsoe,
                             mode='lines', name='Prognoza operatora',
                             line=dict(color='#119DFF', width=2.5, dash='dash')))

    # 3. XGBoost (Vivid Red)
    fig.add_trace(go.Scatter(x=dev_xgb.index, y=dev_xgb,
                             mode='lines', name='XGBoost',
                             line=dict(color='#EF553B', width=2.5)))
                             
    # 4. LightGBM  (Purple)
    fig.add_trace(go.Scatter(x=dev_lgb.index, y=dev_lgb,
                             mode='lines', name='LightGBM',
                             line=dict(color='#AB63FA', width=2.5)))

    fig.update_layout(
        title=dict(text=f'<b>{title}</b>', font=dict(size=18)),
        xaxis_title=dict(text='Data', font=dict(size=14)),
        yaxis_title=dict(text='Błąd predykcji [MW]', font=dict(size=14)),
        font=dict(size=12),
        width=850,
        height=580,
        hovermode='x unified',
        template='plotly_white',
        legend=dict(
            orientation="h", 
            yanchor="top", y=-0.25,
            xanchor="center", x=0.5,
            font=dict(size=12), 
            bgcolor="rgba(255,255,255,0.9)", 
            bordercolor="black", borderwidth=1
        ),
        margin=dict(l=60, r=40, t=60, b=120)
    )

    fig.add_hline(y=0, line_width=1, line_color="black")
    
    fig.update_xaxes(
        tickformat="%d%b %H:00"
    )
    
    fig.show()

plot_period_residuals('2019-12-24', '2019-12-28', 'Scenariusz A: Załamanie popytu (Boże Narodzenie)')
plot_period_residuals('2019-05-29', '2019-06-02', 'Scenariusz B: Dzień pomostowy (Majówka)')
plot_period_residuals('2019-07-18', '2019-07-22', 'Scenariusz C: Standardowy dzień (Lipiec)')